# Validation Set Visualisation
Renders trajectories, parameter distributions and quality metrics for a `.npz` validation file.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ── Change this path to compare different validation sets ─────────────────────
VAL_PATHS = {
    "original (seed42)": "/bettik/PROJECTS/pr-melissa/cesarpi-ext/poolbased_surrogate_validation/ks_res800_al4pde_exact_lv_seed42_n1500_t100.npz",
    "sub5 (seed43)": "/bettik/PROJECTS/pr-melissa/cesarpi-ext/poolbased_surrogate_validation/ks_res800_al4pde_sub5_seed43_n1500_t100.npz",
}
ACTIVE = "sub5 (seed43)"   # which one to use for detailed plots
# ──────────────────────────────────────────────────────────────────────────────

datasets = {}
for name, path in VAL_PATHS.items():
    p = Path(path)
    if p.exists():
        data = np.load(p)
        datasets[name] = {
            "trajectories": data["trajectories"],  # [N, T+1, 1, X]
            "params": data["params"],
            "states0": data["states0"],
        }
        N, T1, _, X = datasets[name]["trajectories"].shape
        print(f"{name}: {N} trajectories × {T1-1} steps, resolution={X}")
    else:
        print(f"⚠ {name}: file not found at {path}")

assert ACTIVE in datasets, f"{ACTIVE!r} not loaded"
traj = datasets[ACTIVE]["trajectories"]   # [N, T+1, 1, X]
params = datasets[ACTIVE]["params"]        # [N, P]
N, T1, _, X = traj.shape
T = T1 - 1
print(f"\nUsing: {ACTIVE}  —  {N} traj, {T} steps, res={X}")

## Summary statistics

In [ ]:
flat = traj.reshape(-1)
print(f"value: mean={flat.mean():.4f}  std={flat.std():.4f}")
print(f"       p1={np.quantile(flat,0.01):.3f}  p99={np.quantile(flat,0.99):.3f}")
tv = np.mean(np.sum(np.abs(np.diff(traj[:, :, 0, :], axis=-1)), axis=-1))
print(f"total variation (mean over all steps): {tv:.1f}")

fft = np.fft.rfft(traj[:, :, 0, :].reshape(-1, X), axis=-1)
power = np.abs(fft)**2
cutoff = X // 4
hf_ratio = power[:, cutoff:].sum(axis=1) / (power.sum(axis=1) + 1e-8)
print(f"high-freq energy ratio (>25%): mean={hf_ratio.mean():.4f}")

# NaN check
finite_ratio = np.isfinite(traj).all(axis=(1,2,3)).mean()
print(f"finite trajectories: {finite_ratio*100:.1f}%")

## Spacetime plots — random sample of trajectories

In [ ]:
rng = np.random.default_rng(0)
n_show = 8
idx = rng.choice(N, size=n_show, replace=False)

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
vmax = float(np.quantile(np.abs(traj[idx]), 0.99))

for ax, i in zip(axes.flat, idx):
    # traj[i] shape: [T+1, 1, X] → use traj[i, :, 0, :] → [T+1, X]
    im = ax.imshow(
        traj[i, :, 0, :].T,       # shape [X, T+1], x on y-axis, t on x-axis
        aspect="auto",
        origin="lower",
        cmap="RdBu_r",
        vmin=-vmax, vmax=vmax,
        extent=[0, T, 0, 64],
    )
    p = params[i]
    ax.set_title(f"#{i}  p=[{p[0]:.2f},{p[1]:.1f}]", fontsize=9)
    ax.set_xlabel("time step")
    ax.set_ylabel("x")

fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.6, label="u(x,t)")
fig.suptitle(f"{ACTIVE} — spacetime plots (x=space, t=time)", fontsize=12)
plt.tight_layout()
plt.savefig("validation_spacetime.png", dpi=120, bbox_inches="tight")
plt.show()

## Snapshots at t=0, t=T/2, t=T for a few trajectories

In [ ]:
n_show = 4
idx2 = rng.choice(N, size=n_show, replace=False)
x = np.linspace(0, 64, X)
timesteps = [0, T // 4, T // 2, T]

fig, axes = plt.subplots(n_show, len(timesteps), figsize=(14, 2.5 * n_show), sharex=True)
for row, i in enumerate(idx2):
    ymax = float(np.abs(traj[i, :, 0, :]).max()) * 1.1
    for col, t in enumerate(timesteps):
        ax = axes[row, col]
        ax.plot(x, traj[i, t, 0, :], lw=0.8)
        ax.set_ylim(-ymax, ymax)
        if row == 0:
            ax.set_title(f"t={t}")
        if col == 0:
            ax.set_ylabel(f"traj {i}\np=[{params[i,0]:.2f},{params[i,1]:.1f}]", fontsize=8)
        ax.set_xlabel("x")

fig.suptitle(f"{ACTIVE} — field snapshots", fontsize=11)
plt.tight_layout()
plt.savefig("validation_snapshots.png", dpi=120, bbox_inches="tight")
plt.show()

## Parameter distribution

In [ ]:
param_names = ["ν (viscosity-like)", "L (length-like)"]
fig, axes = plt.subplots(1, params.shape[1], figsize=(5 * params.shape[1], 4))
if params.shape[1] == 1:
    axes = [axes]
for ax, col, name in zip(axes, range(params.shape[1]), param_names):
    ax.hist(params[:, col], bins=40, color="steelblue", edgecolor="white", linewidth=0.3)
    ax.set_xlabel(name)
    ax.set_ylabel("count")
    ax.set_title(f"param[{col}]: mean={params[:,col].mean():.2f}  std={params[:,col].std():.2f}")
fig.suptitle(f"{ACTIVE} — parameter distribution", fontsize=11)
plt.tight_layout()
plt.show()

## Power spectrum (mean over all snapshots)

In [ ]:
states_flat = traj[:, :, 0, :].reshape(-1, X)  # [N*(T+1), X]
spectra = np.abs(np.fft.rfft(states_flat, axis=-1))**2
mean_spectrum = spectra.mean(axis=0)
freqs = np.fft.rfftfreq(X, d=64.0 / X)  # wavenumbers

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(freqs[1:], mean_spectrum[1:], lw=1)
ax.axvline(freqs[X//4], color="red", ls="--", lw=0.8, label="25% cutoff")
ax.set_xlabel("wavenumber k")
ax.set_ylabel("power |û(k)|²")
ax.set_title(f"{ACTIVE} — mean power spectrum")
ax.legend()
plt.tight_layout()
plt.show()

## Side-by-side comparison (if both datasets loaded)

In [ ]:
if len(datasets) < 2:
    print("Only one dataset loaded — skipping comparison")
else:
    fig, axes = plt.subplots(len(datasets), 3, figsize=(14, 4 * len(datasets)))
    if len(datasets) == 1:
        axes = [axes]
    rng2 = np.random.default_rng(7)

    for row, (name, ds) in enumerate(datasets.items()):
        t_ds = ds["trajectories"]
        n_ds = t_ds.shape[0]
        i = rng2.integers(n_ds)
        vmax = float(np.quantile(np.abs(t_ds[i]), 0.99))

        # Spacetime
        axes[row][0].imshow(t_ds[i, :, 0, :].T, aspect="auto", origin="lower",
                            cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        axes[row][0].set_title(f"{name} — spacetime #{i}")
        axes[row][0].set_xlabel("time step"); axes[row][0].set_ylabel("x")

        # Final snapshot
        axes[row][1].plot(t_ds[i, -1, 0, :], lw=0.8)
        axes[row][1].set_title(f"{name} — final snapshot")
        axes[row][1].set_xlabel("x")

        # Power spectrum
        flat = t_ds[:, :, 0, :].reshape(-1, t_ds.shape[-1])
        sp = np.abs(np.fft.rfft(flat, axis=-1))**2
        axes[row][2].semilogy(sp.mean(axis=0)[1:], lw=1)
        axes[row][2].set_title(f"{name} — mean spectrum")
        axes[row][2].set_xlabel("wavenumber")

    plt.tight_layout()
    plt.savefig("validation_comparison.png", dpi=120, bbox_inches="tight")
    plt.show()